# 02 · Quantum mechanics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/notebooks/02_quantum_mechanics.ipynb)

Everything here uses natural units, $\hbar = m = 1$. The time-independent Schrödinger equation is

$$-\tfrac12\,\psi''(x) + V(x)\,\psi(x) = E\,\psi(x).$$

In [ ]:
# Setup: installs the qplay package if it is missing (for example on Google Colab, about 20 s).
import importlib.util, subprocess, sys
if importlib.util.find_spec("qplay") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/StativeSh/quantum-playground"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import quantum

## Any potential, numerically

`solve_tise` diagonalises the finite-difference Hamiltonian. For $V=\tfrac12x^2$ it should return $E_n = n+\tfrac12$.

In [ ]:
x = np.linspace(-6, 6, 1200)
E, psi = quantum.solve_tise(lambda s: 0.5 * s**2, x, n_states=6)
print(np.round(E, 4))

Now a **double well**, $V = (x^2-4)^2/8$, where the two lowest levels split because the particle can tunnel between the wells.

In [ ]:
V = (x**2 - 4) ** 2 / 8
E, psi = quantum.solve_tise(V, x, n_states=4)
plt.plot(x, V, "k", lw=1)
for n in range(4):
    plt.plot(x, E[n] + psi[n]); plt.axhline(E[n], lw=0.4)
plt.ylim(0, 4); plt.title(f"E = {np.round(E, 3)}");

## Tunnelling through a barrier

For $E < V_0$ the exact transmission through a rectangular barrier of width $a$ is

$$T = \left[1 + \frac{V_0^2\sinh^2(\kappa a)}{4E(V_0-E)}\right]^{-1},\qquad \kappa=\sqrt{2m(V_0-E)}/\hbar .$$

In [ ]:
Es = np.linspace(0.05, 4, 400)
for a in (0.5, 1, 2):
    plt.plot(Es, quantum.barrier_transmission(Es, V0=2.0, a=a), label=f"a = {a}")
plt.axvline(2.0, color="k", lw=0.5)
plt.xlabel("E"); plt.ylabel("T"); plt.legend();

## A wavepacket hitting the barrier

`split_step` evolves $\psi(x,t)$ with the split-step Fourier method.

In [ ]:
x = np.linspace(-60, 60, 2048, endpoint=False)
V = np.where(np.abs(x) < 1, 2.0, 0.0)
psi0 = quantum.gaussian_wavepacket(x, x0=-20, sigma=3, k0=1.5)
times, psi = quantum.split_step(psi0, x, V, dt=0.01, steps=3000, save_every=500)
for p, t in zip(psi, times):
    plt.plot(x, np.abs(p) ** 2, label=f"t = {t:.0f}")
plt.xlim(-45, 45); plt.legend(); plt.xlabel("x"); plt.ylabel(r"$|\psi|^2$");

## Spin-1/2 and the Bloch sphere

Under $H=\tfrac{\omega}{2}\sigma_x$ a spin starting in $|\!\uparrow\rangle$ oscillates between up and down (Rabi oscillation).

In [ ]:
omega = 2.0
ts = np.linspace(0, 2 * np.pi / omega, 100)
bloch = np.array([quantum.bloch_vector(quantum.evolve(omega / 2 * quantum.SIGMA_X, [1, 0], t)) for t in ts])
plt.plot(ts, bloch[:, 2], label=r"$\langle\sigma_z\rangle$")
plt.plot(ts, bloch[:, 1], label=r"$\langle\sigma_y\rangle$")
plt.xlabel("t"); plt.legend();

## Your turn

- Change the barrier height or the packet's $k_0$ and compare the simulated transmission with `barrier_transmission`.
- Use `quantum.hydrogen_wavelength_nm` to list the Balmer series.